In [1]:
from langchain_core.documents import Document

from langchain_ollama import OllamaEmbeddings

from langchain_chroma import Chroma

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs = [

    Document(
        page_content="Employees get 20 days of paid annual leaves.",
        metadata={
            "dept": "HR",
            "year": 2025,
            "file_type": "pdf"
        }
    ),

    Document(
        page_content="Maternity leave policy provides 26 weeks of paid time off.",
        metadata={
            "dept": "HR",
            "year": 2023,
            "file_type": "pdf"
        }
    ),

    Document(
        page_content="Server maintenance costs must be pre-approved by the CTO.",
        metadata={
            "dept": "IT",
            "year": 2025,
            "file_type": "docx"
        }
    ),

    Document(
        page_content="Travel reimbursement requires original food receipts and bills.",
        metadata={
            "dept": "Finance",
            "year": 2025,
            "file_type": "pdf"
        }
    )

]

In [3]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [4]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="chroma_metadata_db"
)

In [5]:
query = "Tell me about time off and allowance policies."

In [6]:
print("=" * 80)
print("CASE 1 : HR DOCUMENTS")
print("=" * 80)

results = vectorstore.similarity_search(
    query,
    k=2,
    filter={
        "dept": "HR"
    }
)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)

CASE 1 : HR DOCUMENTS
Maternity leave policy provides 26 weeks of paid time off.
{'year': 2023, 'file_type': 'pdf', 'dept': 'HR'}
--------------------------------------------------
Employees get 20 days of paid annual leaves.
{'year': 2025, 'file_type': 'pdf', 'dept': 'HR'}
--------------------------------------------------


In [7]:
print("=" * 80)
print("CASE 2 : HR + YEAR 2025")
print("=" * 80)

results = vectorstore.similarity_search(
    query,
    k=2,
    filter={
        "$and": [
            {
                "dept": {
                    "$eq": "HR"
                }
            },
            {
                "year": {
                    "$eq": 2025
                }
            }
        ]
    }
)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)

CASE 2 : HR + YEAR 2025
Employees get 20 days of paid annual leaves.
{'file_type': 'pdf', 'year': 2025, 'dept': 'HR'}
--------------------------------------------------


In [8]:
results = vectorstore.similarity_search(
    query,
    filter={
        "file_type": "pdf"
    }
)

for doc in results:
    print(doc.page_content)

Maternity leave policy provides 26 weeks of paid time off.
Employees get 20 days of paid annual leaves.
Travel reimbursement requires original food receipts and bills.


In [9]:
results = vectorstore.similarity_search(
    query,
    filter={
        "year": 2025
    }
)

for doc in results:
    print(doc.page_content)

Employees get 20 days of paid annual leaves.
Travel reimbursement requires original food receipts and bills.
Server maintenance costs must be pre-approved by the CTO.


In [10]:
results = vectorstore.similarity_search(
    query,
    filter={
        "year": {
            "$gt": 2023
        }
    }
)

for doc in results:
    print(doc.page_content)

Employees get 20 days of paid annual leaves.
Travel reimbursement requires original food receipts and bills.
Server maintenance costs must be pre-approved by the CTO.


In [11]:
results = vectorstore.similarity_search(
    query,
    filter={
        "$or": [
            {
                "dept": {
                    "$eq": "HR"
                }
            },
            {
                "dept": {
                    "$eq": "Finance"
                }
            }
        ]
    }
)

for doc in results:
    print(doc.page_content)

Maternity leave policy provides 26 weeks of paid time off.
Employees get 20 days of paid annual leaves.
Travel reimbursement requires original food receipts and bills.


In [ ]:
# Common Chroma Operators
# Operator	Meaning
# $eq	Equal
# $ne	Not Equal
# $gt	Greater Than
# $gte	Greater Than or Equal
# $lt	Less Than
# $lte	Less Than or Equal
# $in	Value exists in a list
# $nin	Value not in a list
# $and	Logical AND
# $or	Logical OR